# chunkedprefill-lab on a T4: how much do long prompts hurt interactive users, and what does the token budget buy?

Qwen2.5-3B-Instruct. Two open-loop streams at once: interactive (64-token prompts, 128-token outputs, 4 req/s) and long-prompt (3,500-token prompts, 32-token outputs, 0.4 req/s). Sweep `max-num-batched-tokens` 512 → 8192 with chunked prefill on, plus 2048 with it off, plus interactive-only baselines. ~10 min install, ~5 min per configuration.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# This repo is private: its wheel arrives as the attached Kaggle dataset chunkedprefill-lab-src (see portfolio automation).
!pip install -q /kaggle/input/chunkedprefill-lab-src/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, chunkedprefill_lab.analysis, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## Configurations and the two streams

In [ ]:
import json, subprocess, time
from servellm.launch import ServerConfig, VllmServer
from chunkedprefill_lab.analysis import server_args

MODEL = "Qwen/Qwen2.5-3B-Instruct"; BASE = "http://127.0.0.1:8000"
CONFIGS = [(f"budget={b}", b, True, True) for b in (512, 1024, 2048, 4096, 8192)]
CONFIGS += [("budget=2048, chunked off", 2048, False, True), ("interactive only, budget=2048", 2048, True, False), ("interactive only, budget=512", 512, True, False)]
DURATION = 60  # seconds per stream per configuration

def stream(tag, kind):
    common = ["tokenbench", "sweep", "--base-url", BASE, "--model", MODEL, "--mode", "open", "--duration", str(DURATION), "--warmup", "0", "--request-timeout", "120"]
    if kind == "interactive":
        args = ["--levels", "4", "--prompt-tokens", "64", "--max-tokens", "128", "--out", f"results/{tag}_interactive.json"]
    else:
        args = ["--levels", "0.4", "--prompt-tokens", "3500", "--max-tokens", "32", "--out", f"results/{tag}_long.json"]
    return subprocess.Popen(common + args, stdout=open(f"results/{tag}_{kind}.stdout", "w"), stderr=subprocess.STDOUT)

def warm_up():
    subprocess.run(["tokenbench", "sweep", "--base-url", BASE, "--model", MODEL, "--out", "results/_warmup.json", "--levels", "1,4,16",
                    "--num-requests", "16", "--warmup", "0", "--prompt-tokens", "64", "--max-tokens", "64", "--request-timeout", "60"], check=True, capture_output=True, timeout=300)
    subprocess.run(["tokenbench", "sweep", "--base-url", BASE, "--model", MODEL, "--out", "results/_warmup2.json", "--levels", "1",
                    "--num-requests", "3", "--warmup", "0", "--prompt-tokens", "3500", "--max-tokens", "16", "--request-timeout", "60"], check=False, capture_output=True, timeout=240)   # may legitimately fail when chunking is off

def level(path):
    try:
        return json.load(open(path))["levels"][0]
    except Exception:
        return None

In [ ]:
runs = []
for name, budget, chunked, with_long in CONFIGS:
    tag = name.replace(" ", "_").replace("=", "").replace(",", "")
    server = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9,
                                     extra_args=tuple(server_args(budget, chunked))), log_path=f"results/startup_{tag}.log")
    rec = {"config": name, "budget": budget, "chunked": chunked, "with_long": with_long}
    try:
        info = server.start(ready_timeout_s=1200)
        rec["kv_cache_tokens"] = info.kv_cache_tokens
        warm_up()
        procs = [stream(tag, "interactive")] + ([stream(tag, "long")] if with_long else [])
        for p in procs:
            try: p.wait(timeout=DURATION + 240)
            except subprocess.TimeoutExpired: p.kill()
        rec["interactive"] = level(f"results/{tag}_interactive.json")
        rec["long"] = level(f"results/{tag}_long.json") if with_long else None
        it, lg = rec["interactive"], rec["long"]
        print(f"{name:32s} ITL p50/p99 {it['itl']['p50']*1000:.1f}/{it['itl']['p99']*1000:.1f} ms" + (f" | long TTFT p50 {lg['ttft']['p50']*1000:.0f} ms" if lg else ""))
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"; print(name, "FAILED:", rec["error"])
        rec["startup_tail"] = open(f"results/startup_{tag}.log").read()[-1500:] if os.path.exists(f"results/startup_{tag}.log") else ""
    finally:
        server.stop(); time.sleep(5)
    runs.append(rec)
    json.dump(runs, open("results/runs.json", "w"), indent=1)

## Results

In [ ]:
from chunkedprefill_lab.analysis import build_rows, render_markdown
from IPython.display import Markdown, display
ok = [r for r in runs if "error" not in r]
rows = build_rows(ok); md = render_markdown(rows)
open("results/comparison.md", "w").write(md); json.dump(rows, open("results/comparison.json", "w"), indent=1)
display(Markdown(md))
for r in runs:
    if "error" in r: print("FAILED:", r["config"], r["error"][:300])

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
sw = [r for r in rows if r["config"].startswith("budget=") and "off" not in r["config"]]
xs = [r["budget"] for r in sw]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for p_, c in (("p50", "tab:blue"), ("p95", "tab:orange"), ("p99", "tab:red")):
    axes[0].plot(xs, [r[f"interactive_itl_{p_}_ms"] for r in sw], marker="o", color=c, label=f"interactive ITL {p_}")
axes[1].plot(xs, [r["long_ttft_p50_ms"] for r in sw], marker="o", label="long-prompt TTFT p50"); axes[1].plot(xs, [r["long_ttft_p95_ms"] for r in sw], marker="o", label="p95")
axes[2].plot(xs, [r["total_tokens_per_s"] for r in sw], marker="o", label="output tokens/s, both streams")
for ax, t, yl in ((axes[0], "Interference on interactive users", "ms"), (axes[1], "Price paid by long prompts", "ms"), (axes[2], "Throughput", "tokens / s")):
    ax.set_xscale("log", base=2); ax.set_xticks(xs); ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("max-num-batched-tokens"); ax.set_ylabel(yl); ax.set_title(t); ax.grid(alpha=.3); ax.legend()
fig.suptitle("Chunked prefill budget sweep, Qwen2.5-3B on a T4"); fig.tight_layout(); fig.savefig("results/budget_sweep.png", dpi=130)
from IPython.display import Image; display(Image("results/budget_sweep.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip